<a href="https://colab.research.google.com/github/Yash-Yelave/Global_Gatway_RS/blob/main/Data_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies

In [1]:
!pip install groq newspaper3k feedparser requests beautifulsoup4 pandas pydantic lxml_html_clean -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 33.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.7 MB/s eta 0:00:00


Imports & config


In [2]:
import os
import time
import json
import hashlib
import sqlite3
import feedparser
import pandas as pd
import requests

from datetime import datetime
from bs4 import BeautifulSoup
from newspaper import Article
from groq import Groq
from pydantic import BaseModel, field_validator
from typing import Optional, List
from google.colab import userdata

# ── Groq client ──────────────────────────────────────────────────────────────
# In Colab: Secrets (key icon) → add GROQ_API_KEY
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

GROQ_MODEL     = "llama-3.1-8b-instant"
MAX_TOKENS_IN  = 2000          # truncate article before sending to Groq
RATE_LIMIT_SEC = 2.5           # stay safely under 30 req/min free tier
CATEGORIES     = ["technology", "politics", "sports", "business",
                  "health", "science", "entertainment", "general"]

RSS feed sources

In [3]:
RSS_FEEDS = {
    "BBC Top Stories"       : "http://feeds.bbci.co.uk/news/rss.xml",
    "BBC Technology"        : "http://feeds.bbci.co.uk/news/technology/rss.xml",
    "Reuters Top News"      : "https://feeds.reuters.com/reuters/topNews",
    "TechCrunch"            : "https://techcrunch.com/feed/",
    "The Guardian World"    : "https://www.theguardian.com/world/rss",
    "The Guardian Tech"     : "https://www.theguardian.com/uk/technology/rss",
    "Al Jazeera"            : "https://www.aljazeera.com/xml/rss/all.xml",
    "NASA Breaking News"    : "https://www.nasa.gov/rss/dyn/breaking_news.rss",
    "ESPN Top Headlines"    : "https://www.espn.com/espn/rss/news",
    "Harvard Health"        : "https://www.health.harvard.edu/blog/feed",
}

def fetch_rss_urls(feeds: dict, max_per_feed: int = 10) -> list[dict]:
    """
    Parse each RSS feed and return a flat list of
    {title, url, published, source} dicts.
    """
    articles = []
    for source_name, feed_url in feeds.items():
        try:
            feed = feedparser.parse(feed_url)
            entries = feed.entries[:max_per_feed]
            for entry in entries:
                url = entry.get("link", "")
                if not url:
                    continue
                articles.append({
                    "source"   : source_name,
                    "url"      : url,
                    "rss_title": entry.get("title", ""),
                    "published": entry.get("published", ""),
                })
            print(f"  ✓ {source_name}: {len(entries)} URLs")
        except Exception as e:
            print(f"  ✗ {source_name}: {e}")
    return articles

print("Fetching RSS feeds...")
raw_urls = fetch_rss_urls(RSS_FEEDS, max_per_feed=10)
print(f"\nTotal URLs collected: {len(raw_urls)}")

Fetching RSS feeds...
  ✓ BBC Top Stories: 10 URLs
  ✓ BBC Technology: 10 URLs
  ✓ Reuters Top News: 0 URLs
  ✓ TechCrunch: 10 URLs
  ✓ The Guardian World: 10 URLs
  ✓ The Guardian Tech: 10 URLs
  ✓ Al Jazeera: 10 URLs
  ✓ NASA Breaking News: 10 URLs
  ✓ ESPN Top Headlines: 10 URLs
  ✓ Harvard Health: 0 URLs

Total URLs collected: 80


Article scraper

In [5]:
def scrape_article(url: str) -> dict | None:
    """
    Use newspaper3k to pull full article text, author, and publish date.
    Falls back to BeautifulSoup for the meta description if needed.
    Returns None on failure.
    """
    try:
        art = Article(url, fetch_images=False, request_timeout=10)
        art.download()
        art.parse()

        # BeautifulSoup fallback for meta description
        meta_desc = ""
        try:
            soup = BeautifulSoup(art.html, "html.parser")
            tag = (soup.find("meta", attrs={"name": "description"}) or
                   soup.find("meta", attrs={"property": "og:description"}))
            if tag:
                meta_desc = tag.get("content", "")
        except Exception:
            pass

        return {
            "url"          : url,
            "scraped_title": art.title or "",
            "raw_text"     : art.text or "",
            "scraped_author": ", ".join(art.authors) if art.authors else "",
            "scraped_date" : str(art.publish_date) if art.publish_date else "",
            "meta_desc"    : meta_desc,
            "top_image"    : art.top_image or "",
        }
    except Exception as e:
        print(f"    Scrape failed [{url[:60]}]: {e}")
        return None

Groq extraction prompt

In [6]:
EXTRACTION_PROMPT = """
You are a structured data extractor for a news recommendation system.

Given the raw text of a news article, return ONLY a valid JSON object — no explanation, no markdown fences.

JSON schema:
{{
  "title"       : "clean headline (string)",
  "description" : "2-3 sentence plain-English summary (string)",
  "author"      : "author name or null",
  "tags"        : ["keyword1", "keyword2", "keyword3"],
  "category"    : "one of: technology | politics | sports | business | health | science | entertainment | general",
  "sentiment"   : "positive | neutral | negative",
  "publish_date": "YYYY-MM-DD or null"
}}

Rules:
- tags must be 3 to 5 short lowercase keywords relevant to the article.
- description must be written in your own words, not copied from the article.
- If a field cannot be determined, use null.
- Return ONLY the JSON object, nothing else.

Article title (hint): {title_hint}
Article text:
{article_text}
""".strip()


def extract_with_groq(scraped: dict) -> dict | None:
    """
    Send truncated article text to Groq and parse the returned JSON.
    """
    raw_text = scraped.get("raw_text", "")
    if not raw_text.strip():
        raw_text = scraped.get("meta_desc", "")
    if not raw_text.strip():
        return None

    # Truncate to keep within context window
    truncated = raw_text[:MAX_TOKENS_IN * 4]   # ~4 chars per token estimate

    prompt = EXTRACTION_PROMPT.format(
        title_hint   = scraped.get("scraped_title", ""),
        article_text = truncated,
    )

    try:
        response = client.chat.completions.create(
            model       = GROQ_MODEL,
            messages    = [{"role": "user", "content": prompt}],
            max_tokens  = 512,
            temperature = 0.1,
        )
        raw_json = response.choices[0].message.content.strip()

        # Strip accidental markdown fences if present
        if raw_json.startswith("```"):
            raw_json = raw_json.split("```")[1]
            if raw_json.startswith("json"):
                raw_json = raw_json[4:]

        return json.loads(raw_json)
    except json.JSONDecodeError as e:
        print(f"    JSON parse error: {e}")
        return None
    except Exception as e:
        print(f"    Groq API error: {e}")
        return None

Pydantic schema for validation

In [7]:
class NewsArticle(BaseModel):
    article_id  : str
    url         : str
    source      : str
    title       : str
    description : str
    author      : Optional[str]
    tags        : List[str]
    category    : str
    sentiment   : str
    publish_date: Optional[str]
    top_image   : Optional[str]
    scraped_at  : str

    @field_validator("category")
    @classmethod
    def validate_category(cls, v):
        return v if v in CATEGORIES else "general"

    @field_validator("sentiment")
    @classmethod
    def validate_sentiment(cls, v):
        return v if v in ("positive", "neutral", "negative") else "neutral"

    @field_validator("tags")
    @classmethod
    def validate_tags(cls, v):
        return [t.lower().strip() for t in v if t.strip()][:5]

    @field_validator("title", "description")
    @classmethod
    def not_empty(cls, v):
        if not v or not v.strip():
            raise ValueError("Field cannot be empty")
        return v.strip()


def build_article_id(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest()[:12]


def validate_and_merge(rss_meta: dict, scraped: dict, extracted: dict) -> NewsArticle | None:
    """
    Merge RSS metadata + scraped data + Groq extracted fields into
    a validated NewsArticle. Returns None if required fields are missing.
    """
    try:
        return NewsArticle(
            article_id  = build_article_id(rss_meta["url"]),
            url         = rss_meta["url"],
            source      = rss_meta["source"],
            title       = extracted.get("title") or scraped.get("scraped_title") or rss_meta.get("rss_title", ""),
            description = extracted.get("description", ""),
            author      = extracted.get("author") or scraped.get("scraped_author") or None,
            tags        = extracted.get("tags", []),
            category    = extracted.get("category", "general"),
            sentiment   = extracted.get("sentiment", "neutral"),
            publish_date= extracted.get("publish_date") or scraped.get("scraped_date") or rss_meta.get("published") or None,
            top_image   = scraped.get("top_image") or None,
            scraped_at  = datetime.utcnow().isoformat(),
        )
    except Exception as e:
        print(f"    Validation error: {e}")
        return None

Main pipeline loop

In [8]:
def run_pipeline(raw_urls: list[dict], max_articles: int = 50) -> list[dict]:
    """
    Full pipeline: scrape → Groq extract → validate.
    Returns a list of clean article dicts ready for saving.
    """
    results    = []
    seen_urls  = set()
    processed  = 0

    for item in raw_urls:
        if processed >= max_articles:
            break

        url = item["url"]

        # Deduplicate by URL
        if url in seen_urls:
            continue
        seen_urls.add(url)

        print(f"[{processed + 1}/{max_articles}] {url[:80]}")

        # Step 1: Scrape
        scraped = scrape_article(url)
        if not scraped:
            continue

        # Step 2: Groq extraction
        extracted = extract_with_groq(scraped)
        if not extracted:
            continue

        # Step 3: Validate & merge
        article = validate_and_merge(item, scraped, extracted)
        if not article:
            continue

        results.append(article.model_dump())
        processed += 1
        print(f"    ✓ [{article.category}] [{article.sentiment}] {article.title[:60]}")

        # Rate limiting
        time.sleep(RATE_LIMIT_SEC)

    print(f"\nPipeline complete. {len(results)} clean articles collected.")
    return results


# Run — set max_articles to however many you want
clean_articles = run_pipeline(raw_urls, max_articles=50)

[1/50] https://www.bbc.com/news/articles/c895jpwl9gpo?at_medium=RSS&at_campaign=rss


/tmp/ipykernel_4716/3272300405.py:60: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scraped_at  = datetime.utcnow().isoformat(),


    ✓ [politics] [neutral] dangerous moment for keir starmer
[2/50] https://www.bbc.com/news/articles/cjd84pkkjgpo?at_medium=RSS&at_campaign=rss
    ✓ [business] [neutral] unemployment rate unexpectedly falls
[3/50] https://www.bbc.com/news/articles/c5ylw4q5jv2o?at_medium=RSS&at_campaign=rss
    ✓ [general] [negative] drive-by shooting outside Harlesden church
[4/50] https://www.bbc.com/news/articles/cn08jy6w0l5o?at_medium=RSS&at_campaign=rss
    ✓ [health] [neutral] uk smoking ban for people born after 2008 agreed
[5/50] https://www.bbc.com/news/articles/crm19j48wnno?at_medium=RSS&at_campaign=rss
    ✓ [entertainment] [neutral] madonna offers reward for missing coachella costume
[6/50] https://www.bbc.com/news/articles/cn9qw1jqpgxo?at_medium=RSS&at_campaign=rss
    ✓ [politics] [neutral] No 10 considered giving Doyle ambassador role, sacked offici
[7/50] https://www.bbc.com/news/articles/cd9v420y190o?at_medium=RSS&at_campaign=rss
    ✓ [politics] [negative] zelensky criticizes us envo

Save to CSV, JSON, and SQLite

In [9]:
def save_outputs(articles: list[dict], prefix: str = "news_dataset"):
    df = pd.DataFrame(articles)

    # ── CSV ──────────────────────────────────────────────────────────────────
    csv_path = f"{prefix}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved CSV  → {csv_path}  ({len(df)} rows)")

    # ── JSON (records orient) ─────────────────────────────────────────────────
    json_path = f"{prefix}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON → {json_path}")

    # ── SQLite ───────────────────────────────────────────────────────────────
    db_path = f"{prefix}.db"
    conn    = sqlite3.connect(db_path)

    df_sql = df.copy()
    # SQLite can't store Python lists — serialise tags to JSON string
    df_sql["tags"] = df_sql["tags"].apply(json.dumps)
    df_sql.to_sql("articles", conn, if_exists="replace", index=False)

    conn.commit()
    conn.close()
    print(f"Saved DB   → {db_path}  (table: articles)")

    return df

df = save_outputs(clean_articles)
df.head()

Saved CSV  → news_dataset.csv  (50 rows)
Saved JSON → news_dataset.json
Saved DB   → news_dataset.db  (table: articles)


,article_id,url,source,title,description,author,tags,category,sentiment,publish_date,top_image,scraped_at
0,40dc511eb4a1,https://www.bbc.com/news/articles/c895jpwl9gpo...,BBC Top Stories,dangerous moment for keir starmer,labour party's judgment on the prime minister ...,None,"[labour, politics, uk]",politics,neutral,"Tue, 21 Apr 2026 12:16:18 GMT",https://ichef.bbci.co.uk/news/1024/branded_new...,2026-04-21T12:30:37.150789
1,795becaf0ccc,https://www.bbc.com/news/articles/cjd84pkkjgpo...,BBC Top Stories,unemployment rate unexpectedly falls,the unemployment rate has decreased due to few...,None,"[unemployment, economy, war]",business,neutral,"Tue, 21 Apr 2026 11:20:19 GMT",https://ichef.bbci.co.uk/news/1024/branded_new...,2026-04-21T12:30:40.263711
2,26e07c3ba064,https://www.bbc.com/news/articles/c5ylw4q5jv2o...,BBC Top Stories,drive-by shooting outside Harlesden church,A drive-by shooting occurred outside a church ...,None,"[shooting, murder, harlesden]",general,negative,"Tue, 21 Apr 2026 11:49:29 GMT",https://ichef.bbci.co.uk/news/1024/branded_new...,2026-04-21T12:30:43.238631
3,129e95f24cfe,https://www.bbc.com/news/articles/cn08jy6w0l5o...,BBC Top Stories,uk smoking ban for people born after 2008 agreed,the uk government has agreed to implement a sm...,None,"[uk, smoking ban, health]",health,neutral,"Tue, 21 Apr 2026 11:39:52 GMT",https://ichef.bbci.co.uk/news/1024/branded_new...,2026-04-21T12:30:46.255826
4,53fca67a3c96,https://www.bbc.com/news/articles/crm19j48wnno...,BBC Top Stories,madonna offers reward for missing coachella co...,madonna is offering a reward for the return of...,None,"[madonna, coachella, costume, reward]",entertainment,neutral,"Tue, 21 Apr 2026 10:29:05 GMT",https://ichef.bbci.co.uk/news/1024/branded_new...,2026-04-21T12:30:49.254757


Quick EDA (optional sanity check)

In [10]:
print("=== Dataset summary ===\n")
print(f"Total articles : {len(df)}")
print(f"Sources        : {df['source'].nunique()}")
print(f"Date range     : {df['publish_date'].min()} → {df['publish_date'].max()}")

print("\n── Category distribution ──")
print(df["category"].value_counts().to_string())

print("\n── Sentiment distribution ──")
print(df["sentiment"].value_counts().to_string())

print("\n── Articles per source ──")
print(df["source"].value_counts().to_string())

print("\n── Null counts ──")
print(df.isnull().sum().to_string())

=== Dataset summary ===

Total articles : 50
Sources        : 6
Date range     : 2024-04-10 → null

── Category distribution ──
category
technology       28
politics         10
general           4
entertainment     3
business          2
health            1
sports            1
science           1

── Sentiment distribution ──
sentiment
neutral     30
negative    18
positive     2

── Articles per source ──
source
BBC Top Stories       10
TechCrunch            10
The Guardian Tech     10
The Guardian World    10
BBC Technology         9
Al Jazeera             1

── Null counts ──
article_id       0
url              0
source           0
title            0
description      0
author          22
tags             0
category         0
sentiment        0
publish_date     0
top_image        0
scraped_at       0


Download files from Colab


In [11]:
from google.colab import files

files.download("news_dataset.csv")
files.download("news_dataset.json")
files.download("news_dataset.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>